# 02 · Preprocesamiento ECoG por bloque

Este notebook genera los datos del **paso 2** para S1/S2/S3 a partir de los `.mat` de desarrollo:
exclusión fija de canales → referencia de mediana común → FIR 40–200 Hz + notch 60/120/180 Hz → Morlet (40 frecuencias, 7 ciclos) → antialias y decimación a 100 Hz; guante a 25 Hz → spline cúbico a 100 Hz; pares ECoG(t) → flexión(t + 200 ms); folds con escaladores ajustados solo con su train.

**Reglas**
- **Unidad independiente = bloque.** B1–B5 se procesan una vez cada uno y solo con su propia ECoG y su propio guante; ninguna característica depende de otro bloque y ninguna ventana cruza fronteras entre bloques.
- **Compuerta del EDA.** Antes de procesar se valida `channels.reference_run`: corrida `COMPLETE`, mismos hashes de los `.mat`, misma exclusión y señal, y figuras de evidencia intactas.
- **Purga pendiente.** `purge_samples_at_100hz` sigue en `null`: los folds se construyen con el candidato de verificación del YAML y quedan etiquetados `candidate_pending_V04_V09` en todos los manifiestos. Al aprobarse la purga (V04/V09) se regeneran sin reprocesar los bloques.
- Las etiquetas de test **nunca** se leen. Nada de lo que muestra este notebook está aprobado hasta completar el paso 3.

**Salidas en Drive**
- `MyDrive/TT_ECoG/data/processed/<pipeline_hash>/`: `S*/B*/` (características, flexión, índices y máscaras), `manifests/` y `folds/purge_<P>_<hash>/`.
- `MyDrive/TT_ECoG/results/preprocessing/<run_id>/`: evidencia de esta corrida (tablas, figuras, manifiestos).

Si el repositorio es privado, autentica Git de forma interactiva; no escribas tokens en celdas ni URLs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

## Entorno
Se instalan solo las dependencias que falten (`requirements-eda.txt`: incluye MNE y scikit-learn). No se instala PyTorch ni CUDA. Si un módulo cambia después de importarlo, reinicia el runtime.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/check_eda_environment.py'), '--install-missing'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '->', tt_ecog.__file__)

## Drive y nueva corrida
Las rutas salen del YAML. Cada ejecución crea una carpeta de evidencia nueva; los datos procesados se identifican por contenido y, si ya existen y su manifiesto verifica, se reutilizan sin sobrescribir.

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
from tt_ecog.eda import show
from tt_ecog.prep import finish_run, load_subjects, start_run, steps
CONFIG = PROJECT / 'configs/experiment_config.yaml'
ctx = start_run(CONFIG, DRIVE_ROOT)
print('Corrida:', ctx.run_id)
print('Evidencia en Drive:', ctx.output)
print('Commit registrado:', ctx.env['git_commit'], '| cambios sin commitear:', ctx.env['git_dirty'])

## Datos de desarrollo
Se leen solo `train_data` y `train_dg`, con hash antes y después de la lectura.

In [ ]:
records = load_subjects(ctx)
ctx.subject_status

## p01 · Compuerta del EDA
Si `channels.reference_run` es `null` o no valida, la celda se detiene y lista las corridas EDA válidas para estos archivos: fija una en el YAML, haz commit/push y vuelve a ejecutar.

In [ ]:
show(steps.p01_gate(ctx, records))

## p02 · Bloques procesados
Cadena completa por bloque. Las columnas válidas exigen todo el contexto FIR→Morlet→antialias dentro del bloque (125 muestras por borde a 100 Hz) y flexión interpolada en t + 20. La potencia negativa por el rizado del antialias se cuenta y no se recorta.

In [ ]:
show(steps.p02_blocks(ctx, records))

## p03 · Folds y escaladores
Para cada sujeto y bloque de validación: columnas de train elegibles (válidas y fuera de la purga), RobustScaler(10, 90) y MinMax ajustados **solo** con ellas, ventanas de entrenamiento (stride 1) y de inferencia (stride 64 + ventana final) dentro de cada tramo.

In [ ]:
show(steps.p03_folds(ctx, records))
print('Purga:', ctx.purge, '|', ctx.purge_status)

## Cierre
Escribe `run_summary.json`, `index.md` y `artifact_manifest.json`; falla si algún paso no terminó en `COMPLETE`.

In [ ]:
summary = finish_run(ctx)
print(summary['status'], '| datos:', summary['pipeline_dir'])
print('folds:', summary['fold_set_dir'], '|', summary['purge_status'])